<a href="https://colab.research.google.com/github/Shaheenovic/Drone-parking-monitoring-yolov8/blob/main/notebooks/02_training_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install -q ultralytics pyyaml

In [15]:
!git clone https://github.com/Shaheenovic/Drone-parking-monitoring-yolov8.git
%cd Drone-parking-monitoring-yolov8
!git status

Cloning into 'Drone-parking-monitoring-yolov8'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 49 (delta 18), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 24.53 KiB | 4.91 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/Drone-parking-monitoring-yolov8/Drone-parking-monitoring-yolov8
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [16]:
from pathlib import Path
import urllib.request
import zipfile

OWNER = "Shaheenovic"
REPO = "Drone-parking-monitoring-yolov8"
TAG = "v1.0"
DATASET_ASSET = "drone-parking-v1-yolo11.zip"

url = f"https://github.com/{OWNER}/{REPO}/releases/download/{TAG}/{DATASET_ASSET}"

archive_path = Path("data") / DATASET_ASSET
extract_path = Path("data/raw")

archive_path.parent.mkdir(parents=True, exist_ok=True)
extract_path.mkdir(parents=True, exist_ok=True)

print("Downloading:", url)
urllib.request.urlretrieve(url, archive_path)

with zipfile.ZipFile(archive_path, "r") as z:
    z.extractall(extract_path)

print("Downloaded to:", archive_path)
print("Extracted to:", extract_path)

Downloading: https://github.com/Shaheenovic/Drone-parking-monitoring-yolov8/releases/download/v1.0/drone-parking-v1-yolo11.zip
Downloaded to: data/drone-parking-v1-yolo11.zip
Extracted to: data/raw


In [18]:
from pathlib import Path
import hashlib
import shutil
import yaml

EXPECTED_SHA256 = "3F9E38E8AE7F7725F19F5F120165A1F19E8A54808D27690648A530CFB6B12999".lower()

sha256 = hashlib.sha256()

with open(archive_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

actual_sha256 = sha256.hexdigest()

print("Expected SHA256:", EXPECTED_SHA256)
print("Actual SHA256:  ", actual_sha256)

if actual_sha256 != EXPECTED_SHA256:
    raise ValueError("SHA256 mismatch: أعد تنزيل ملف الـdataset ولا تبدأ التدريب.")

print("SHA256 verified successfully.")

yaml_files = list(extract_path.rglob("data.yaml"))

if not yaml_files:
    raise FileNotFoundError("لم يتم العثور على data.yaml بعد فك الضغط.")

print("\nFound data.yaml files:")
for p in yaml_files:
    print("-", p)

DATA_YAML = yaml_files[0].resolve()
DATASET_ROOT = DATA_YAML.parent.resolve()

print("\nDataset root:", DATASET_ROOT)
print("\nOriginal data.yaml:\n")
print(DATA_YAML.read_text())

Expected SHA256: 3f9e38e8ae7f7725f19f5f120165a1f19e8a54808d27690648a530cfb6b12999
Actual SHA256:   3f9e38e8ae7f7725f19f5f120165a1f19e8a54808d27690648a530cfb6b12999
SHA256 verified successfully.

Found data.yaml files:
- data/raw/data.yaml

Dataset root: /content/Drone-parking-monitoring-yolov8/Drone-parking-monitoring-yolov8/data/raw

Original data.yaml:

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 4
names: ['Empty', 'Illegal', 'LicensePlate', 'Occupied']

roboflow:
  workspace: eng-ahmed_shaheen-hotmail-com
  project: drone-parking-monitoring-yolov8
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/eng-ahmed_shaheen-hotmail-com/drone-parking-monitoring-yolov8/dataset/1


In [19]:
with open(DATA_YAML, "r") as f:
    data_config = yaml.safe_load(f)

data_config["path"] = str(DATASET_ROOT)

for split in ["train", "val", "test"]:
    if split in data_config:
        print(f"{split}: {data_config[split]}")

FIXED_DATA_YAML = Path("data") / "data_colab.yaml"

with open(FIXED_DATA_YAML, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print("\nSaved corrected config to:", FIXED_DATA_YAML)
print("\nCorrected data.yaml:\n")
print(FIXED_DATA_YAML.read_text())

train: ../train/images
val: ../valid/images
test: ../test/images

Saved corrected config to: data/data_colab.yaml

Corrected data.yaml:

train: ../train/images
val: ../valid/images
test: ../test/images
nc: 4
names:
- Empty
- Illegal
- LicensePlate
- Occupied
roboflow:
  workspace: eng-ahmed_shaheen-hotmail-com
  project: drone-parking-monitoring-yolov8
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/eng-ahmed_shaheen-hotmail-com/drone-parking-monitoring-yolov8/dataset/1
path: /content/Drone-parking-monitoring-yolov8/Drone-parking-monitoring-yolov8/data/raw



In [20]:
from pathlib import Path

for split in ["train", "valid", "val", "test"]:
    image_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    if image_dir.exists():
        images = list(image_dir.glob("*.*"))
        labels = list(label_dir.glob("*.txt")) if label_dir.exists() else []

        print(f"{split}:")
        print(f"  Images: {len(images)}")
        print(f"  Labels: {len(labels)}")

train:
  Images: 401
  Labels: 401
valid:
  Images: 94
  Labels: 94
